---
title: Research community
description: >-
  How many Scopus author profiles publish with a DECE affiliation, how the community renews and
  how long newcomers stay, how concentrated output is and how often DECE researchers lead the
  papers they co-author (aggregate statistics only).
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "dece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from dece_stats import config, data
from dece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the export, partial
AS_OF = config.EXPORT_DATE      # the export date: the data run to this day

In [ ]:
# Aggregate statistics only: author ids serve to count distinct profiles and are never shown.
import math

import plotly.express as px
from dece_stats import metrics, periods

pubs = data.load_publications()
authors = data.load_publication_authors()

FIRST_YEAR = int(pubs["year"].min())
LAST_FULL = periods.LAST_FULL                 # the latest full year (PARTIAL - 1)
P_RECENT = periods.period_of(LAST_FULL)       # the latest five full years
P_BASE = periods.period_of(LAST_FULL - 20)    # the baseline period, twenty years earlier
RECENT_FROM = periods.bounds(P_RECENT)[0]
BASE_YEAR = periods.bounds(P_BASE)[0]         # earlier single year quoted next to the latest full year


def join_and(items: list[str]) -> str:
    """'a', 'a and b', 'a, b and c'."""
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


pubs["period"] = pubs["year"].map(lambda y: periods.period_of(y, first_year=FIRST_YEAR))
pubs["n_other"] = pubs["n_authors"] - pubs["n_dece_authors"]   # co-authors without a DECE affiliation
full = pubs[pubs["year"] < PARTIAL]                            # complete years only
per_year = full.groupby("year").size().reindex(range(FIRST_YEAR, PARTIAL), fill_value=0)
MIN_PAPERS = 20                                                # yearly ratios need enough papers
MIN_YEAR = int(per_year[per_year < MIN_PAPERS].index.max()) + 1   # from here on every year has >= 20
SPARSE_UNTIL = 1994                                            # the record is sparse before the mid-1990s
assert per_year.loc[:SPARSE_UNTIL].mean() < per_year.loc[RECENT_FROM:].mean() / 10, \
    "text: the record is sparse before the mid-1990s"
n_pubs = len(pubs)

# --- DECE author profiles ------------------------------------------------------------------------
dece = authors[authors["is_dece"]]                 # one row per (paper, DECE-affiliated author)
dece_base = dece[dece["year"].between(*periods.bounds(P_BASE))]
dece_recent = dece[dece["year"].between(*periods.bounds(P_RECENT))]
n_profiles = dece["author_id"].nunique()
n_authorships = len(dece)
first_year = dece.groupby("author_id")["year"].min()           # year of each profile's first DECE paper
active = dece[["author_id", "year"]].drop_duplicates()         # one row per active profile and year
active["since"] = active["year"] - active["author_id"].map(first_year)

# active profiles per year by years since the first DECE paper (0 = newcomers)
SENIORITY = ["0 (newcomers)", "1–4", "5–9", "10+"]
active["seniority"] = pd.cut(active["since"], [-1, 0, 4, 9, np.inf], labels=SENIORITY).astype(str)
community = (active.groupby(["year", "seniority"]).size().unstack(fill_value=0)
             .reindex(columns=SENIORITY, fill_value=0))
community["active"] = community[SENIORITY].sum(axis=1)
newcomers = community["0 (newcomers)"]
new_share = newcomers / community["active"]
senior_share = community["10+"] / community["active"]

active_base = int(community.loc[BASE_YEAR, "active"])
active_last = int(community.loc[LAST_FULL, "active"])
newcomers_recent = int(newcomers.loc[RECENT_FROM:LAST_FULL].sum())
newcomers_last = int(newcomers.loc[LAST_FULL])
new_share_lo, new_share_hi = new_share.loc[RECENT_FROM:LAST_FULL].min(), new_share.loc[RECENT_FROM:LAST_FULL].max()
active_recent = dece_recent["author_id"].nunique()
SENIOR_FROM = LAST_FULL - 14                                   # the latest fifteen full years
senior_lo, senior_hi = senior_share.loc[SENIOR_FROM:LAST_FULL].min(), senior_share.loc[SENIOR_FROM:LAST_FULL].max()
senior_last = int(community.loc[LAST_FULL, "10+"])
assert active_last > active_base, "text: the number of active profiles rose"
assert new_share_lo >= 0.2, "text: the community renews quickly (newcomers are a fifth or more of every recent year)"

# papers per active profile, by year and pooled over 5-year periods
per_profile = pd.DataFrame({
    "authorships": dece.groupby("year").size(),          # a paper counts once per DECE author
    "papers": dece.groupby("year")["eid"].nunique(),     # papers with at least one DECE author
    "active": community["active"],
})
per_profile["full"] = per_profile["authorships"] / per_profile["active"]
per_profile["dept"] = per_profile["papers"] / per_profile["active"]
pp_period = (per_profile.loc[:LAST_FULL]
             .groupby(lambda y: periods.period_of(y, first_year=FIRST_YEAR))[["authorships", "papers", "active"]]
             .sum())
pp_period["full"] = pp_period["authorships"] / pp_period["active"]
pp_period["dept"] = pp_period["papers"] / pp_period["active"]
assert (pp_period.loc[P_RECENT, ["full", "dept"]] > pp_period.loc[P_BASE, ["full", "dept"]]).all(), \
    "text: output grew faster than the community (both ratios rose)"

# growth between the two compared periods (five complete years each), which unlike a comparison
# of two single years does not hinge on the choice of the end years
active_mean_early = pp_period.loc[P_BASE, "active"] / periods.WIDTH     # active profiles a year, on average
active_mean_recent = pp_period.loc[P_RECENT, "active"] / periods.WIDTH
active_growth = pp_period.loc[P_RECENT, "active"] / pp_period.loc[P_BASE, "active"]
papers_early, papers_recent = int(pp_period.loc[P_BASE, "papers"]), int(pp_period.loc[P_RECENT, "papers"])
papers_growth = papers_recent / papers_early
assert 1 < active_growth < papers_growth, "text: the community grew, and output grew faster"

# --- newcomer cohorts and retention ---------------------------------------------------------------
FOLLOW = 5   # years of follow-up after the first DECE paper
# cohorts: the 5-year periods from the baseline on whose newcomers can be followed for FOLLOW complete
# years (the latest is the period before P_RECENT)
COHORTS = [p for p in periods.periods(FIRST_YEAR)
           if periods.bounds(p)[0] >= BASE_YEAR and periods.bounds(p)[1] + FOLLOW <= LAST_FULL]
C_FIRST, C_LAST = COHORTS[0], COHORTS[-1]                                   # the oldest and the latest cohort
COHORT_SPAN = f"{periods.bounds(C_FIRST)[0]}–{periods.bounds(C_LAST)[1]}"   # all cohorts together
EXAMPLE_FIRST = periods.bounds(C_LAST)[0]      # example in the caption: a first DECE paper in this year


def cohort_ids(lo: int, hi: int) -> pd.Index:
    """Profiles whose first DECE paper appeared in the years lo..hi."""
    return first_year.index[first_year.between(lo, hi)]


def returned(ids: pd.Index, lo: int, hi: int) -> float:
    """Share of the profiles `ids` with a DECE paper lo..hi years after their first one."""
    hit = active["author_id"].isin(ids) & active["since"].between(lo, hi)
    return active.loc[hit, "author_id"].nunique() / len(ids)


rows = []
for label in COHORTS:
    lo, hi = periods.bounds(label)
    assert hi + FOLLOW <= LAST_FULL, "every cohort needs five complete years of follow-up"
    ids = cohort_ids(lo, hi)
    rows.append({"cohort": label, "n": len(ids),
                 "within": returned(ids, 1, FOLLOW), "year5": returned(ids, FOLLOW, FOLLOW)})
retention = pd.DataFrame(rows).set_index("cohort")
cohort_n = retention["n"]
cohort_n_text = join_and([dp.fmt_int(n) for n in cohort_n])
within_first, within_last = retention.loc[C_FIRST, "within"], retention.loc[C_LAST, "within"]
year5_first, year5_last = retention.loc[C_FIRST, "year5"], retention.loc[C_LAST, "year5"]
ids_all = cohort_ids(*periods.bounds(COHORT_SPAN))
n_cohort_all = len(ids_all)
ret1_all = returned(ids_all, 1, 1)
within_all = returned(ids_all, 1, FOLLOW)
ret5_all = returned(ids_all, FOLLOW, FOLLOW)
assert within_all < 0.5 and ret5_all < 0.25, "text: few newcomers stay; most publish with DECE only briefly"
assert retention["year5"].between(0.15, 0.25).all(), "text: the fifth-year share stayed at about one in five"
assert within_last < within_first, "text: the share coming back within five years fell"


def any_later(lo: int, hi: int, until: int = LAST_FULL) -> float:
    """Share of a cohort with any DECE paper five or more years after the first (up to `until`)."""
    ids = cohort_ids(lo, hi)
    hit = active["author_id"].isin(ids) & active["since"].ge(FOLLOW) & active["year"].le(until)
    return active.loc[hit, "author_id"].nunique() / len(ids)


later_early, later_recent = any_later(*periods.bounds(C_FIRST)), any_later(*periods.bounds(C_LAST))
# the oldest cohort given only as many years in which to return as the latest one had
lag = periods.bounds(C_LAST)[0] - periods.bounds(C_FIRST)[0]
later_early_same = any_later(*periods.bounds(C_FIRST), until=LAST_FULL - lag)
assert later_early - later_early_same > later_early_same - later_recent, \
    "text: the gap is mainly because the older cohort had more years in which to return"

# --- productivity distribution and concentration --------------------------------------------------
papers_per_profile = dece.groupby("author_id")["eid"].nunique()    # all years
BIN_LABELS = ["1", "2", "3–4", "5–9", "10–19", "20–49", "50–99", "100+"]
bins = pd.cut(papers_per_profile, [0, 1, 2, 4, 9, 19, 49, 99, np.inf], labels=BIN_LABELS)
dist = papers_per_profile.groupby(bins, observed=False).agg(profiles="size", authorships="sum")
assert dist["profiles"].min() >= 10, "privacy: every bin must hold at least 10 profiles"
dist["share_profiles"] = dist["profiles"] / dist["profiles"].sum()
dist["share_authorships"] = dist["authorships"] / dist["authorships"].sum()
one_paper_share = (papers_per_profile == 1).mean()
assert abs(one_paper_share - 0.5) < 0.05, "text: about half of all profiles have a single DECE paper"
# the same share for profiles whose first DECE paper left them FOLLOW or more full years to add more
OLD_UNTIL = LAST_FULL - FOLLOW
one_paper_share_old = (papers_per_profile[first_year.reindex(papers_per_profile.index) <= OLD_UNTIL] == 1).mean()
assert one_paper_share_old < one_paper_share, "text: the single-paper share is partly a matter of timing"
median_papers = papers_per_profile.median()
median_text = (f"{dp.fmt_num(median_papers, 0 if median_papers == int(median_papers) else 1)} "
               f"DECE paper{'' if median_papers == 1 else 's'}")
top_bin_profiles = int(dist.loc["100+", "profiles"])
top_bin_share = dist.loc["100+", "share_authorships"]


def top_tenth(rows: pd.DataFrame, papers: pd.DataFrame) -> dict:
    """The most prolific tenth of the profiles in `rows` (at or above the 90th percentile of DECE
    papers, ties included): their number, the threshold, and the shares of `papers` they appear on
    and of the DECE authorships in `rows` they hold."""
    counts = rows.groupby("author_id")["eid"].nunique()
    threshold = math.ceil(counts.quantile(0.9))
    top = counts.index[counts >= threshold]
    return {"n_top": len(top), "threshold": threshold,
            "paper_share": rows.loc[rows["author_id"].isin(top), "eid"].nunique() / len(papers),
            "authorship_share": counts[top].sum() / counts.sum()}


top_all = top_tenth(dece, pubs)
top_early = top_tenth(dece_base, full[full["period"] == P_BASE])
top_recent = top_tenth(dece_recent, full[full["period"] == P_RECENT])


def concentration_of(rows: pd.DataFrame) -> dict:
    """Measures that do not grow with the number of profiles alone: the Gini coefficient of DECE
    papers per profile (all profiles, and profiles with two or more papers), the share of the DECE
    authorships held by the 20 most prolific profiles together, the DECE authors per paper and the
    share of the profiles with a single DECE paper in `rows` (occasional contributors)."""
    counts = rows.groupby("author_id")["eid"].nunique()
    return {"gini": metrics.gini(counts), "gini2": metrics.gini(counts[counts >= 2]),
            "top20": counts.nlargest(20).sum() / counts.sum(), "per_paper": rows.groupby("eid").size().mean(),
            "single": (counts == 1).mean()}


conc_early = concentration_of(dece_base)
conc_recent = concentration_of(dece_recent)
checks_conc = {
    "output is concentrated: the top tenth of P_RECENT appear on most of its papers": top_recent["paper_share"] > 0.5,
    "the Gini coefficient rose (all profiles and two or more papers)":
        conc_recent["gini"] > conc_early["gini"] and conc_recent["gini2"] > conc_early["gini2"],
    "the top decile's shares of papers and authorships rose":
        top_recent["paper_share"] > top_early["paper_share"]
        and top_recent["authorship_share"] > top_early["authorship_share"],
    "the 20 most prolific hold a similar share": abs(conc_recent["top20"] - conc_early["top20"]) < 0.05,
    "more occasional contributors": conc_recent["single"] > conc_early["single"],
    "papers carry more DECE authors": conc_recent["per_paper"] > conc_early["per_paper"],
}
assert all(checks_conc.values()), f"text: {[k for k, ok in checks_conc.items() if not ok]}"

# --- leadership positions by 5-year period (complete years) ---------------------------------------
multi = full[full["n_authors"] >= 2]                    # the last author differs from the first
known = (full[full["dece_corresponding"].notna()]       # papers with a usable corresponding address
         .assign(corr=lambda d: d["dece_corresponding"].astype(bool)))
lead = pd.DataFrame({
    "first": full.groupby("period")["dece_first_author"].mean(),
    "first_n": full.groupby("period").size(),
    "last": multi.groupby("period")["dece_last_author"].mean(),
    "last_n": multi.groupby("period").size(),
    "corr": known.groupby("period")["corr"].mean(),
    "corr_n": known.groupby("period").size(),
    "first_or_last": (multi["dece_first_author"] | multi["dece_last_author"]).groupby(multi["period"]).mean(),
}).sort_index()                                                   # labels start with the year: chronological
lead["corr_cov"] = lead["corr_n"].fillna(0) / lead["first_n"]
LEAD_MIN = 50                                                     # papers (or addresses) for a period to be shown
lead_shown = lead[lead["first_n"] >= LEAD_MIN].copy()
lead_shown.loc[lead_shown["corr_n"].fillna(0) < LEAD_MIN, "corr"] = np.nan
first_early, first_recent = lead.loc[P_BASE, "first"], lead.loc[P_RECENT, "first"]
last_early, last_recent = lead.loc[P_BASE, "last"], lead.loc[P_RECENT, "last"]
corr_early, corr_recent = lead.loc[P_BASE, "corr"], lead.loc[P_RECENT, "corr"]
fl_early, fl_recent = lead.loc[P_BASE, "first_or_last"], lead.loc[P_RECENT, "first_or_last"]
corr_first_agree = (known["corr"] == known["dece_first_author"]).mean()
# gap between the two series in the periods the text compares (from P_BASE on, where addresses are many)
corr_gap = (lead_shown["corr"] - lead_shown["first"]).abs()[lead_shown.index >= P_BASE].max()
# the period since the baseline with the fewest usable corresponding addresses
P_LOW = lead.loc[lead.index >= P_BASE, "corr_cov"].idxmin()
conf_low = full[(full["period"] == P_LOW) & (full["source_kind"] == "conference")]
corr_cov_conf_low = conf_low["dece_corresponding"].notna().mean()
corr_cov_low, corr_cov_recent = lead.loc[P_LOW, "corr_cov"], lead.loc[P_RECENT, "corr_cov"]
corr_missing = full["dece_corresponding"].isna().mean()
checks_lead = {
    "the chart ends with P_RECENT": lead_shown.index[-1] == P_RECENT,
    "the DECE first-author share fell": first_recent < first_early,
    "the last-author share fell less": 0 < last_early - last_recent < first_early - first_recent,
    "the corresponding-author share follows the first-author share closely":
        corr_first_agree > 0.85 and corr_gap < 0.1 and (corr_recent < corr_early) == (first_recent < first_early),
    "many papers lack a usable corresponding address": corr_missing > 0.2,
    "few addresses in P_LOW, fewer still for its conference papers, more in P_RECENT":
        P_LOW != P_RECENT and corr_cov_conf_low < corr_cov_low < corr_cov_recent,
}
assert all(checks_lead.values()), f"text: {[k for k, ok in checks_lead.items() if not ok]}"

# first authorship by collaboration type (institution level), P_BASE vs P_RECENT
TYPES = {"single-institution": "Single-institution", "national": "National", "international": "International"}
two = full[full["period"].isin([P_BASE, P_RECENT])]
type_n = pd.crosstab(two["collab_type"], two["period"]).reindex(list(TYPES))
type_mix = type_n / type_n.sum()
type_first = two.pivot_table(index="collab_type", columns="period", values="dece_first_author",
                             aggfunc="mean").reindex(list(TYPES))
type_last = (two[two["n_authors"] >= 2]
             .pivot_table(index="collab_type", columns="period", values="dece_last_author", aggfunc="mean")
             .reindex(list(TYPES)))
first_cf = (type_mix[P_RECENT] * type_first[P_BASE]).sum()   # the P_RECENT mix at the P_BASE rates
mix_pts = (first_early - first_cf) * 100
total_pts = (first_early - first_recent) * 100
# the split depends on the reference rates: the mix effect evaluated at the P_RECENT rates instead
mix_pts_alt = ((type_mix[P_BASE] - type_mix[P_RECENT]) * type_first[P_RECENT]).sum() * 100


def shown_pct(share: float) -> float:
    """A share in percent as the text prints it (dp.fmt_pct, one decimal)."""
    return float(dp.fmt_pct(share).rstrip("%").replace("−", "-").replace(",", ""))


# the points the text quotes next to the shares are differences of the shares as printed, so that
# they add up for readers (each is within 0.1 point of the unrounded difference used in the guards)
total_pts_shown = shown_pct(first_early) - shown_pct(first_recent)
mix_pts_shown = shown_pct(first_early) - shown_pct(first_cf)
NAT_INTL = ["national", "international"]
checks_type = {
    "single-institution papers almost always have a DECE first author": type_first.loc["single-institution"].min() > 0.9,
    "the single-institution share fell":
        type_mix.loc["single-institution", P_RECENT] < type_mix.loc["single-institution", P_BASE],
    "the international share rose": type_mix.loc["international", P_RECENT] > type_mix.loc["international", P_BASE],
    "DECE authors are first less often on international papers":
        (type_first.loc["international"] < type_first.drop(index="international").min()).all(),
    "the first-author share fell within national and international papers":
        (type_first.loc[NAT_INTL, P_RECENT] < type_first.loc[NAT_INTL, P_BASE]).all(),
    "the mix explains part of the decline (at either reference rates)":
        0 < mix_pts < total_pts and 0 < mix_pts_alt < total_pts,
    "the printed points are part of the printed decline":
        0 < mix_pts_shown < total_pts_shown and 0 < round(mix_pts_alt, 1) < total_pts_shown,
    "the DECE last-author share of international papers rose":
        type_last.loc["international", P_RECENT] > type_last.loc["international", P_BASE],
}
assert all(checks_type.values()), f"text: {[k for k, ok in checks_type.items() if not ok]}"

# --- co-authorship inside the department ----------------------------------------------------------
team = full[full["year"] >= MIN_YEAR].groupby("year").agg(       # full years only
    dece=("n_dece_authors", "mean"), other=("n_other", "mean"),
    ge2=("n_dece_authors", lambda s: s.ge(2).mean()),
    only_dece=("n_other", lambda s: s.eq(0).mean()))
team_p = full.groupby("period").agg(
    dece=("n_dece_authors", "mean"), other=("n_other", "mean"),
    ge2=("n_dece_authors", lambda s: s.ge(2).mean()),
    only_dece=("n_other", lambda s: s.eq(0).mean()),
    has_other=("n_other", lambda s: s.gt(0).mean()))
grew_other = team_p.loc[P_RECENT, "other"] - team_p.loc[P_BASE, "other"]
grew_dece = team_p.loc[P_RECENT, "dece"] - team_p.loc[P_BASE, "dece"]
checks_team = {
    "teams grew mostly through other co-authors; DECE authors per paper rose only a little":
        grew_other > grew_dece > 0,
    "more papers have a co-author without a DECE affiliation":
        team_p.loc[P_RECENT, "has_other"] > team_p.loc[P_BASE, "has_other"],
    "most papers (still) have two or more DECE authors": (team_p.loc[[P_BASE, P_RECENT], "ge2"] > 0.5).all(),
    "papers by DECE authors alone fell": team_p.loc[P_RECENT, "only_dece"] < team_p.loc[P_BASE, "only_dece"],
}
assert all(checks_team.values()), f"text: {[k for k, ok in checks_team.items() if not ok]}"
others_recent = authors[~authors["is_dece"] & authors["year"].between(*periods.bounds(P_RECENT))]
other_dece_elsewhere = others_recent["author_id"].isin(first_year.index).mean()
# "occasionally, DECE members whose affiliation on that paper does not name the department": the
# non-DECE authorships of P_RECENT whose profile has a DECE paper in the same year
other_dece_same_year = (others_recent.merge(active[["author_id", "year"]], how="left", indicator=True)
                        ["_merge"].eq("both").mean())
assert other_dece_same_year < 0.1, "text: only occasionally do DECE members not name the department"
n_no_dece = int((pubs["n_dece_authors"] == 0).sum())
no_dece_text = ("Every record has at least one author identified as DECE." if n_no_dece == 0 else
                f"In {dp.fmt_int(n_no_dece)} record{'' if n_no_dece == 1 else 's'} no author is identified as DECE.")

This page looks at the people behind the department's publications as a community: how many researchers publish with a DECE affiliation, how the community renews and how long newcomers stay, how concentrated output is, how often DECE researchers hold the first, last and corresponding author positions, and how they write papers together. Researchers are counted as Scopus author profiles, and an author counts as DECE on a paper when their own affiliation on that paper names the department (see [Notes & caveats](#people-notes)). All numbers are aggregates: the page shows no names, identifiers or per-person figures. Across the {eval}`dp.fmt_int(n_pubs)` records, {eval}`dp.fmt_int(n_profiles)` distinct profiles appear with a DECE affiliation at least once, in {eval}`dp.fmt_int(n_authorships)` DECE authorships (paper–author pairs).

In [ ]:
dp.kpi_html([
    dp.KPI("DECE author profiles", n_profiles, f"distinct Scopus profiles, {FIRST_YEAR}–{PARTIAL}"),
    dp.KPI(f"Active in {LAST_FULL}", active_last,
           f"profiles with a DECE paper ({dp.fmt_int(active_base)} in {BASE_YEAR})"),
    dp.KPI("Newcomers active in year five", dp.fmt_pct(ret5_all),
           f"first DECE paper in {COHORT_SPAN} ({dp.fmt_int(n_cohort_all)} profiles)"),
    dp.KPI("Publications with a DECE first author", dp.fmt_pct(first_recent),
           f"{P_RECENT} ({dp.fmt_pct(first_early)} in {P_BASE})"),
], min_width=140)

:::{important} Key findings
- An average of {eval}`dp.fmt_int(active_mean_recent)` author profiles a year published with a DECE affiliation in {eval}`P_RECENT`, {eval}`dp.fmt_num(active_growth, 1)` times as many as in {eval}`P_BASE` ({eval}`dp.fmt_int(active_mean_early)`). Output grew faster: {eval}`dp.fmt_int(papers_recent)` papers in {eval}`P_RECENT` against {eval}`dp.fmt_int(papers_early)` in {eval}`P_BASE` ({eval}`dp.fmt_num(papers_growth, 1)` times as many).
- The community renews quickly, but few newcomers stay: of the {eval}`dp.fmt_int(n_cohort_all)` profiles whose first DECE paper appeared in {eval}`COHORT_SPAN`, {eval}`dp.fmt_pct(within_all)` published with a DECE affiliation again within five years and {eval}`dp.fmt_pct(ret5_all)` in the fifth year after the first. The share coming back within five years fell from {eval}`dp.fmt_pct(within_first)` for the newcomers of {eval}`C_FIRST` to {eval}`dp.fmt_pct(within_last)` for those of {eval}`C_LAST`.
- Output is concentrated: {eval}`dp.fmt_pct(one_paper_share)` of all profiles have a single DECE paper, while the most prolific tenth of the profiles active in {eval}`P_RECENT` ({eval}`dp.fmt_int(top_recent['n_top'])` profiles) appear on {eval}`dp.fmt_pct(top_recent['paper_share'])` of the department's papers of those years.
- A DECE author is first author on {eval}`dp.fmt_pct(first_recent)` of the papers of {eval}`P_RECENT`, down from {eval}`dp.fmt_pct(first_early)` in {eval}`P_BASE`. Part of the decline reflects the rise of international papers (from {eval}`dp.fmt_pct(type_mix.loc['international', P_BASE])` to {eval}`dp.fmt_pct(type_mix.loc['international', P_RECENT])` of output), on which DECE authors are first less often: with the {eval}`P_BASE` rate of each collaboration type, the {eval}`P_RECENT` mix alone gives {eval}`dp.fmt_pct(first_cf)`.
- Teams grew mostly through co-authors without a DECE affiliation: their number per paper rose from {eval}`dp.fmt_num(team_p.loc[P_BASE, 'other'], 1)` in {eval}`P_BASE` to {eval}`dp.fmt_num(team_p.loc[P_RECENT, 'other'], 1)` in {eval}`P_RECENT`, while the number of DECE-affiliated authors per paper rose only from {eval}`dp.fmt_num(team_p.loc[P_BASE, 'dece'], 1)` to {eval}`dp.fmt_num(team_p.loc[P_RECENT, 'dece'], 1)`.
:::

## Size of the research community

An author profile is *active* in a year when it has at least one paper with a DECE affiliation in that year. [](#fig-people-active) splits each year's active profiles by the time since the profile's first DECE paper; *newcomers* are profiles whose first DECE paper appeared in that year.

In [ ]:
#| label: cell-people-active
active_tidy = (community.loc[MIN_YEAR:, SENIORITY].rename_axis(columns="since")
               .stack().rename("profiles").reset_index()
               .merge(community[["active"]], left_on="year", right_index=True))
active_tidy["share"] = active_tidy["profiles"] / active_tidy["active"]
fig = dp.year_bar(active_tidy, "year", "profiles", color="since",
                  colors=dict(zip(SENIORITY, dp.ordinal_colors(len(SENIORITY)))),
                  partial_year=PARTIAL, as_of=AS_OF, value_title="Active author profiles",
                  legend_title="Years since first DECE paper:", custom_data=["share", "active"])
fig.update_traces(hovertemplate="%{y:,.0f} (%{customdata[0]:.0%} of %{customdata[1]:,})")
fig.show()

:::{figure} #cell-people-active
:label: fig-people-active
Scopus author profiles with at least one DECE-affiliated paper in each year, {eval}`str(MIN_YEAR)`–{eval}`str(PARTIAL)`, by years since the profile's first DECE paper (0 = newcomers). The record starts in {eval}`str(FIRST_YEAR)` and is sparse before the mid-1990s, so in the early years the longer-standing groups are understated. {eval}`str(PARTIAL)` is partial (data to {eval}`AS_OF.strftime('%d %B').lstrip('0')`).
:::

The number of active profiles rose from {eval}`dp.fmt_int(active_base)` in {eval}`str(BASE_YEAR)` to {eval}`dp.fmt_int(active_last)` in {eval}`str(LAST_FULL)`, the latest full year ([](#fig-people-active)), and from an average of {eval}`dp.fmt_int(active_mean_early)` a year in {eval}`P_BASE` to {eval}`dp.fmt_int(active_mean_recent)` in {eval}`P_RECENT`. The community renews steadily: {eval}`dp.fmt_int(newcomers_recent)` profiles published their first DECE paper in {eval}`P_RECENT` ({eval}`dp.fmt_int(newcomers_last)` of them in {eval}`str(LAST_FULL)`), and newcomers made up between {eval}`dp.fmt_pct(new_share_lo)` and {eval}`dp.fmt_pct(new_share_hi)` of the active profiles in each of these years. Altogether {eval}`dp.fmt_int(active_recent)` distinct profiles published with a DECE affiliation in {eval}`P_RECENT`. Profiles whose first DECE paper was ten or more years earlier made up between {eval}`dp.fmt_pct(senior_lo)` and {eval}`dp.fmt_pct(senior_hi)` of the active profiles in every full year from {eval}`str(SENIOR_FROM)` to {eval}`str(LAST_FULL)` ({eval}`dp.fmt_int(senior_last)` of the {eval}`dp.fmt_int(active_last)` in {eval}`str(LAST_FULL)`).

### Papers per active profile

In [ ]:
#| label: cell-people-per-profile
MEASURES = {"full": "DECE authorships per active profile", "dept": "Papers per active profile"}
pp_tidy = (per_profile.loc[MIN_YEAR:LAST_FULL, list(MEASURES)].rename(columns=MEASURES)   # full years only
           .rename_axis(columns="measure").stack().rename("value").reset_index())
fig = dp.year_line(pp_tidy, "year", "value", color="measure",
                   value_title="Per active profile and year", value_format=".2f")
fig.update_yaxes(rangemode="tozero")
fig.show()

:::{figure} #cell-people-per-profile
:label: fig-people-per-profile
DECE authorships and papers per active DECE author profile, {eval}`str(MIN_YEAR)`–{eval}`str(LAST_FULL)` (from {eval}`str(MIN_YEAR)` on every year has at least {eval}`str(MIN_PAPERS)` papers). *DECE authorships per active profile* counts a paper once for each of its DECE authors; *papers per active profile* divides the number of papers with a DECE author by the number of active profiles. {eval}`str(PARTIAL)` (partial year) is not shown.
:::

Output grew faster than the community ([](#fig-people-per-profile)). In {eval}`P_RECENT` an active profile accounted for {eval}`dp.fmt_num(pp_period.loc[P_RECENT, 'full'], 2)` DECE authorships a year on average ({eval}`dp.fmt_num(pp_period.loc[P_BASE, 'full'], 2)` in {eval}`P_BASE`), and the department published {eval}`dp.fmt_num(pp_period.loc[P_RECENT, 'dept'], 2)` papers a year per active profile ({eval}`dp.fmt_num(pp_period.loc[P_BASE, 'dept'], 2)` in {eval}`P_BASE`). Both ratios also depend on what Scopus indexes, for example how completely it covers each year's conference proceedings.

## Newcomers and retention

To see how long newcomers stay, profiles are grouped by the period of their first DECE paper (cohorts) and followed for the next five years; a profile counts as returning only with another paper that carries a DECE affiliation. Every cohort up to {eval}`C_LAST` can be followed for five complete years (to {eval}`str(LAST_FULL)` at the latest), so all cohorts are compared over the same follow-up.

In [ ]:
#| label: cell-people-retention
OUTCOMES = {"within": "Another DECE paper within five years", "year5": "A DECE paper in the fifth year"}
ret_tidy = (retention[list(OUTCOMES)].rename(columns=OUTCOMES).rename_axis(columns="outcome")
            .stack().rename("share").reset_index()
            .merge(retention[["n"]], left_on="cohort", right_index=True))
fig = px.bar(ret_tidy, x="cohort", y="share", color="outcome", barmode="group", custom_data=["n"],
             category_orders={"cohort": COHORTS, "outcome": list(OUTCOMES.values())},
             color_discrete_map=dp.color_map(OUTCOMES.values()),
             labels={"cohort": "Newcomers by year of their first DECE paper", "share": "Share of the newcomers"})
fig.update_traces(texttemplate="%{y:.0%}", textposition="outside",
                  hovertemplate="%{y:.1%} of %{customdata[0]:,} newcomers")
fig.update_layout(hovermode="x unified", legend_title_text="", margin=dict(t=24))
fig.update_yaxes(tickformat=".0%", hoverformat=".1%", range=[0, ret_tidy["share"].max() * 1.2])
fig.show()

:::{figure} #cell-people-retention
:label: fig-people-retention
Newcomer cohorts by the period of their first DECE paper ({eval}`cohort_n_text` profiles): share with another DECE paper within the five years after the first, and share with a DECE paper in the fifth year after the first (for example in {eval}`str(EXAMPLE_FIRST + FOLLOW)` for a first paper in {eval}`str(EXAMPLE_FIRST)`). Follow-up ends in {eval}`str(LAST_FULL)`.
:::

Most newcomers publish with the department only briefly ([](#fig-people-retention)). Of the {eval}`dp.fmt_int(n_cohort_all)` profiles whose first DECE paper appeared in {eval}`COHORT_SPAN`, {eval}`dp.fmt_pct(within_all)` published with a DECE affiliation again within five years ({eval}`dp.fmt_pct(ret1_all)` in the very next year) and {eval}`dp.fmt_pct(ret5_all)` in the fifth year after the first. The share still publishing in the fifth year stayed at about one in five ({eval}`dp.fmt_pct(year5_first)` for the newcomers of {eval}`C_FIRST`, {eval}`dp.fmt_pct(year5_last)` for those of {eval}`C_LAST`), but the share coming back at all within five years fell from {eval}`dp.fmt_pct(within_first)` to {eval}`dp.fmt_pct(within_last)`. The pattern is consistent with a stable core of long-term members and a large, rotating group of students, visitors, project staff and external co-authors who list a DECE affiliation; split Scopus profiles (see the notes) would also lower these shares.

## Concentration of output

[](#fig-people-productivity) groups all profiles by their number of DECE papers since {eval}`str(FIRST_YEAR)` and compares each group's share of the profiles with its share of the DECE authorships.

In [ ]:
#| label: cell-people-productivity
DIST = {"profiles": "Author profiles", "authorships": "DECE authorships"}
dist_tidy = pd.concat(
    [pd.DataFrame({"bin": BIN_LABELS, "measure": label,
                   "share": dist[f"share_{col}"].to_numpy(), "count": dist[col].to_numpy()})
     for col, label in DIST.items()], ignore_index=True)
fig = px.bar(dist_tidy, x="bin", y="share", color="measure", barmode="group", custom_data=["count"],
             category_orders={"bin": BIN_LABELS, "measure": list(DIST.values())},
             color_discrete_map=dp.color_map(DIST.values()),
             labels={"bin": "DECE papers per author profile (logarithmic bins)", "share": "Share of the total"})
fig.update_traces(hovertemplate="%{y:.1%} (%{customdata[0]:,})")
fig.update_layout(hovermode="x unified", legend_title_text="", margin=dict(t=24))
fig.update_yaxes(tickformat=".0%", hoverformat=".1%")
fig.show()

:::{figure} #cell-people-productivity
:label: fig-people-productivity
Author profiles by number of DECE papers, {eval}`str(FIRST_YEAR)`–{eval}`str(PARTIAL)`: share of all {eval}`dp.fmt_int(n_profiles)` profiles and share of all {eval}`dp.fmt_int(n_authorships)` DECE authorships (a paper counts once for each of its DECE authors) in each group. The groups widen roughly logarithmically.
:::

About half of all profiles ({eval}`dp.fmt_pct(one_paper_share)`) have a single DECE paper, partly because recent profiles have had little time to add more (see [Notes & caveats](#people-notes)), and the median is {eval}`median_text` ([](#fig-people-productivity)). At the other end, the {eval}`dp.fmt_int(top_bin_profiles)` profiles with 100 or more DECE papers hold {eval}`dp.fmt_pct(top_bin_share)` of all DECE authorships, and the most prolific tenth of profiles ({eval}`dp.fmt_int(top_all['n_top'])` profiles with at least {eval}`dp.fmt_int(top_all['threshold'])` DECE papers) appear on {eval}`dp.fmt_pct(top_all['paper_share'])` of the department's {eval}`dp.fmt_int(n_pubs)` records. By the Gini coefficient of DECE papers per profile ({eval}`dp.fmt_num(conc_early['gini'], 2)` in {eval}`P_BASE`, {eval}`dp.fmt_num(conc_recent['gini'], 2)` in {eval}`P_RECENT`; {eval}`dp.fmt_num(conc_early['gini2'], 2)` and {eval}`dp.fmt_num(conc_recent['gini2'], 2)` among profiles with at least two papers) and by the top decile, output became more concentrated: the most prolific tenth of the profiles active in {eval}`P_RECENT` ({eval}`dp.fmt_int(top_recent['n_top'])` profiles with at least {eval}`dp.fmt_int(top_recent['threshold'])` DECE papers in those years) appear on {eval}`dp.fmt_pct(top_recent['paper_share'])` of the papers of {eval}`P_RECENT` and hold {eval}`dp.fmt_pct(top_recent['authorship_share'])` of the period's DECE authorships, against {eval}`dp.fmt_pct(top_early['paper_share'])` and {eval}`dp.fmt_pct(top_early['authorship_share'])` for the most prolific tenth of {eval}`P_BASE` ({eval}`dp.fmt_int(top_early['n_top'])` profiles). The 20 most prolific profiles of each period, however, hold a similar share of its DECE authorships ({eval}`dp.fmt_pct(conc_early['top20'])` in {eval}`P_BASE`, {eval}`dp.fmt_pct(conc_recent['top20'])` in {eval}`P_RECENT`), so the change reflects a larger number of occasional contributors rather than more output in fewer hands. The share of papers with a top-decile author also rises as papers carry more DECE authors ({eval}`dp.fmt_num(conc_early['per_paper'], 2)` on average in {eval}`P_BASE`, {eval}`dp.fmt_num(conc_recent['per_paper'], 2)` in {eval}`P_RECENT`). Concentration of this kind is expected where research groups form around permanent staff while many students and visitors contribute to a few papers each; it says nothing about individual merit.

## Leadership positions

The first author has usually done most of the work, the last author often leads the group, and the corresponding author handles the submission. These conventions vary between fields and groups, so the shares below are rough indicators of how often DECE researchers lead the papers they co-author.

In [ ]:
#| label: cell-people-leadership
ROLES = {"first": "DECE first author", "last": "DECE last author", "corr": "DECE corresponding author"}
lead_tidy = pd.concat(
    [pd.DataFrame({"period": lead_shown.index, "role": label, "share": lead_shown[col].to_numpy(),
                   "n": lead_shown[f"{col}_n"].to_numpy()})
     for col, label in ROLES.items()], ignore_index=True).dropna(subset=["share"])
fig = px.line(lead_tidy, x="period", y="share", color="role", markers=True, custom_data=["n"],
              category_orders={"period": list(lead_shown.index), "role": list(ROLES.values())},
              color_discrete_map=dp.color_map(ROLES.values()),
              labels={"period": "", "share": "Share of publications"})
fig.update_traces(hovertemplate="%{y:.1%} of %{customdata[0]:,.0f} papers")
fig.update_layout(hovermode="x unified", legend_title_text="", margin=dict(t=24))
fig.update_yaxes(tickformat=".0%", hoverformat=".1%", range=[0, 1])
fig.show()

:::{figure} #cell-people-leadership
:label: fig-people-leadership
Share of papers with a DECE-affiliated first author (all papers), last author (papers with two or more authors) and corresponding author (papers with a usable corresponding address in Scopus), by 5-year period, {eval}`lead_shown.index[0]` to {eval}`P_RECENT`. Periods with fewer than {eval}`str(LEAD_MIN)` papers (or addresses) are not shown, and {eval}`str(PARTIAL)` is left out as partial.
:::

DECE researchers are the first author of {eval}`dp.fmt_pct(first_recent)` of the papers of {eval}`P_RECENT`, down from {eval}`dp.fmt_pct(first_early)` in {eval}`P_BASE` ([](#fig-people-leadership)). The last-author share fell less, from {eval}`dp.fmt_pct(last_early)` to {eval}`dp.fmt_pct(last_recent)`, and a DECE author is first or last on {eval}`dp.fmt_pct(fl_recent)` of the multi-author papers of {eval}`P_RECENT` ({eval}`dp.fmt_pct(fl_early)` in {eval}`P_BASE`). The corresponding-author share ({eval}`dp.fmt_pct(corr_recent)` in {eval}`P_RECENT`, {eval}`dp.fmt_pct(corr_early)` in {eval}`P_BASE`) follows the first-author share closely: where both are known they agree on {eval}`dp.fmt_pct(corr_first_agree)` of the papers. It is the least reliable of the three series, because Scopus lacks a usable corresponding address for many papers: only {eval}`dp.fmt_pct(corr_cov_low)` of the papers of {eval}`P_LOW` have one (conference papers of those years: {eval}`dp.fmt_pct(corr_cov_conf_low)`), against {eval}`dp.fmt_pct(corr_cov_recent)` in {eval}`P_RECENT`.

In [ ]:
#| label: cell-people-first-by-type
by_type_tbl = pd.DataFrame(
    [{"type": label if p == P_BASE else "", "period": p, "papers": int(type_n.loc[key, p]),
      "share": type_mix.loc[key, p], "first": type_first.loc[key, p], "last": type_last.loc[key, p]}
     for key, label in TYPES.items() for p in (P_BASE, P_RECENT)]
    + [{"type": "All papers" if p == P_BASE else "", "period": p, "papers": int(type_n[p].sum()),
        "share": 1.0, "first": lead.loc[p, "first"], "last": lead.loc[p, "last"]} for p in (P_BASE, P_RECENT)])
dp.table_html(by_type_tbl,
              columns={"type": "Type", "period": "Period", "papers": "Papers",
                       "share": "Share", "first": "DECE first", "last": "DECE last"},
              formats={"share": "pct", "first": "pct", "last": "pct"})

:::{figure} #cell-people-first-by-type
:kind: table
:label: tbl-people-first-by-type
Papers by collaboration type (institution level, as on the [Collaboration](collaboration.ipynb) page) in {eval}`P_BASE` and {eval}`P_RECENT`: number of papers, their share of the period's papers, and the share with a DECE-affiliated first author (*DECE first*) and last author (*DECE last*, papers with two or more authors).
:::

The decline goes with a change in how papers are produced ([](#tbl-people-first-by-type)). Single-institution papers almost always have a DECE first author ({eval}`dp.fmt_pct(type_first.loc['single-institution', P_BASE])` in {eval}`P_BASE`, {eval}`dp.fmt_pct(type_first.loc['single-institution', P_RECENT])` in {eval}`P_RECENT`), but their share of output fell from {eval}`dp.fmt_pct(type_mix.loc['single-institution', P_BASE])` to {eval}`dp.fmt_pct(type_mix.loc['single-institution', P_RECENT])`, while international papers, where DECE authors are first less often, rose from {eval}`dp.fmt_pct(type_mix.loc['international', P_BASE])` to {eval}`dp.fmt_pct(type_mix.loc['international', P_RECENT])`. Within national and international papers the DECE first-author share fell as well, from {eval}`dp.fmt_pct(type_first.loc['national', P_BASE])` to {eval}`dp.fmt_pct(type_first.loc['national', P_RECENT])` and from {eval}`dp.fmt_pct(type_first.loc['international', P_BASE])` to {eval}`dp.fmt_pct(type_first.loc['international', P_RECENT])`. Had each collaboration type kept its {eval}`P_BASE` first-author share, the {eval}`P_RECENT` mix alone would have lowered the overall share from {eval}`dp.fmt_pct(first_early)` to {eval}`dp.fmt_pct(first_cf)`, against the actual {eval}`dp.fmt_pct(first_recent)`: evaluated at the {eval}`P_BASE` rates, the change in mix accounts for {eval}`dp.fmt_num(mix_pts_shown, 1)` of the {eval}`dp.fmt_num(total_pts_shown, 1)` percentage points of decline ({eval}`dp.fmt_num(mix_pts_alt, 1)` at the {eval}`P_RECENT` rates), and the changes within the types for the rest. On international papers, DECE authors moved partly from the first to the last position: the DECE last-author share of these papers rose from {eval}`dp.fmt_pct(type_last.loc['international', P_BASE])` to {eval}`dp.fmt_pct(type_last.loc['international', P_RECENT])`, consistent with DECE researchers more often acting as senior partners in international teams. The data show who is listed first or last, not who did the work.

## Co-authorship within the department

Here the authors of each paper are split into DECE-affiliated authors and other co-authors: researchers from other AUTH units or other institutions and, occasionally, DECE members whose affiliation on that paper does not name the department.

In [ ]:
#| label: cell-people-team
GROUPS = {"dece": "DECE-affiliated authors", "other": "Other co-authors"}
team_tidy = (team[list(GROUPS)].rename(columns=GROUPS).rename_axis(columns="group")
             .stack().rename("authors").reset_index())
fig = dp.year_bar(team_tidy, "year", "authors", color="group",
                  colors={GROUPS["dece"]: dp.CATEGORICAL[0], GROUPS["other"]: dp.OTHER_COLOR},
                  value_title="Authors per publication (mean)", value_format=".2f")
fig.show()

:::{figure} #cell-people-team
:label: fig-people-team
Mean number of authors per paper, split into DECE-affiliated authors and other co-authors, {eval}`str(MIN_YEAR)`–{eval}`str(LAST_FULL)` (from {eval}`str(MIN_YEAR)` on every year has at least {eval}`str(MIN_PAPERS)` papers). {eval}`str(PARTIAL)` (partial year) is not shown.
:::

Teams grew mostly through co-authors without a DECE affiliation ([](#fig-people-team)). A paper had {eval}`dp.fmt_num(team_p.loc[P_BASE, 'dece'], 2)` DECE-affiliated authors on average in {eval}`P_BASE` and {eval}`dp.fmt_num(team_p.loc[P_RECENT, 'dece'], 2)` in {eval}`P_RECENT`, while its other co-authors rose from {eval}`dp.fmt_num(team_p.loc[P_BASE, 'other'], 2)` to {eval}`dp.fmt_num(team_p.loc[P_RECENT, 'other'], 2)`. In {eval}`P_RECENT`, {eval}`dp.fmt_pct(team_p.loc[P_RECENT, 'has_other'])` of the papers had at least one co-author without a DECE affiliation, against {eval}`dp.fmt_pct(team_p.loc[P_BASE, 'has_other'])` in {eval}`P_BASE`.

In [ ]:
#| label: cell-people-internal
SHARES = {"ge2": "Two or more DECE authors", "only_dece": "Only DECE authors"}
internal_tidy = (team[list(SHARES)].rename(columns=SHARES).rename_axis(columns="measure")
                 .stack().rename("share").reset_index())
fig = dp.year_line(internal_tidy, "year", "share", color="measure",
                   value_title="Share of publications", value_format=".1%")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

:::{figure} #cell-people-internal
:label: fig-people-internal
Share of papers with at least two DECE-affiliated authors and share of papers whose authors all have a DECE affiliation, {eval}`str(MIN_YEAR)`–{eval}`str(LAST_FULL)`. {eval}`str(PARTIAL)` (partial year) is not shown.
:::

Most papers still bring several DECE researchers together: {eval}`dp.fmt_pct(team_p.loc[P_RECENT, 'ge2'])` of the papers of {eval}`P_RECENT` have at least two DECE authors, against {eval}`dp.fmt_pct(team_p.loc[P_BASE, 'ge2'])` in {eval}`P_BASE` ([](#fig-people-internal)). Papers written by DECE authors alone, however, fell from {eval}`dp.fmt_pct(team_p.loc[P_BASE, 'only_dece'])` to {eval}`dp.fmt_pct(team_p.loc[P_RECENT, 'only_dece'])` of output, in line with the growth of collaboration described on the [Collaboration](collaboration.ipynb) page.

(people-notes)=
## Notes & caveats

- **Profiles are not people.** All counts are of Scopus author profiles. Scopus sometimes splits one researcher's papers over two or more profiles or merges namesakes into one; splits inflate the numbers of profiles, newcomers and one-paper profiles and lower the retention shares.
- **Who counts as DECE.** An author counts as DECE on a paper when their own affiliation on that paper has a word starting with "elec" and one starting with "aristot" in the same affiliation string, or an "elec…" department string without an institution name next to an Aristotle University string of the same author (see [Methodology & data](methodology.ipynb)). Authors who give only "Aristotle University of Thessaloniki", or only another institution, are not counted, so one person can be DECE on some papers and not on others: {eval}`dp.fmt_pct(other_dece_elsewhere)` of the non-DECE authorships of {eval}`P_RECENT` belong to profiles that carry a DECE affiliation on other papers (for example earlier or later members, or members who listed only another institution on that paper). {eval}`no_dece_text`
- **Profiles are not staff.** Profiles include PhD students, post-docs, visitors and external collaborators who listed a DECE affiliation, so the counts are not staff numbers, and the distribution of output is not a measure of individual performance. The page deliberately shows no names, identifiers or per-person figures.
- **Newcomers and retention.** A newcomer is a profile with its first DECE paper in this record, which is not the start of employment: the newcomers of the first years include established staff, and people may have published earlier without a DECE affiliation. A profile counts as present in a year only if it has a DECE paper in that year: people may stay without publishing, or leave and keep publishing elsewhere. Counting any DECE paper five or more years after the first (up to {eval}`str(LAST_FULL)`) gives {eval}`dp.fmt_pct(later_early)` for the newcomers of {eval}`C_FIRST` but {eval}`dp.fmt_pct(later_recent)` for those of {eval}`C_LAST`, mainly because the older cohort had more years in which to return; [](#fig-people-retention) therefore uses the same five-year follow-up for every cohort.
- **Recent profiles.** Profiles whose first DECE paper is recent have had little time to add more, so the share of single-paper profiles ({eval}`dp.fmt_pct(one_paper_share)`) is partly a matter of timing: among the profiles whose first DECE paper appeared in {eval}`str(OLD_UNTIL)` or earlier it is {eval}`dp.fmt_pct(one_paper_share_old)`.
- **Author positions.** First, last and corresponding positions are imperfect signals of leadership (alphabetical order and other conventions exist). Scopus has no usable corresponding address for {eval}`dp.fmt_pct(corr_missing)` of the papers of {eval}`str(FIRST_YEAR)`–{eval}`str(LAST_FULL)`, and an address counts as DECE only when it names the department.
- **Partial year.** {eval}`str(PARTIAL)` (data to {eval}`AS_OF.strftime('%d %B %Y').lstrip('0')`) is included in the counts over the whole record, {eval}`str(FIRST_YEAR)`–{eval}`str(PARTIAL)` (records, profiles, DECE authorships and each profile's number of DECE papers, as in [](#fig-people-productivity)), in the shares drawn from them, such as the single-paper shares, and in the yearly count of active profiles ([](#fig-people-active)). It is left out of the other yearly series, of the 5-year periods and newcomer cohorts, and of all comparisons over time.